# OxtaMem — prova de qualidade end-to-end (T4)

**Pergunta:** o modelo NSOS *responde melhor* quando o OxtaMem lhe entrega o fato
relevante, numa tarefa onde os fatos **excedem a janela de contexto**?

Mesmo modelo treinado em recall associativo (MQAR). Unica diferenca no teste:
**modelo-so** (janela recente de C fatos) vs **modelo+OxtaMem** (recupera o fato
certo da memoria O(N) via `forward_trunk` e injeta no contexto).

Runtime: **T4** (Runtime -> Change runtime type -> GPU). Roda as celulas em ordem.

## 1 — GPU + clone do repo (branch `nsos-gpu-phases12`)

In [ ]:
import os, subprocess, pathlib, glob, sys
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'sem GPU? troque o runtime para T4'

REPO = '/content/reimagined'
BRANCH = 'nsos-gpu-phases12'
# repo privado: cole a URL COM token se precisar, ex: https://<TOKEN>@github.com/vitorGgC569/reimagined.git
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/vitorGgC569/reimagined.git')
if not pathlib.Path(REPO).exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,REPO], check=True)
os.environ['REPO'] = REPO
print('repo pronto em', REPO, '| branch', BRANCH)

## 2 — Build do `nsos_ext` (CUDA, arch nativa do runtime)

In [ ]:
BUILD = f'{REPO}/OXN/nsos/build-t4'
subprocess.run(['cmake','-S',f'{REPO}/OXN/nsos','-B',BUILD,'-G','Ninja',
  '-DCMAKE_BUILD_TYPE=Release','-DNSOS_ENABLE_CUDA=ON','-DNSOS_BUILD_PYTHON=ON',
  '-DNSOS_BUILD_TESTS=OFF','-DNSOS_BUILD_CLI=OFF','-DNSOS_BUILD_API=OFF',
  '-DCMAKE_CUDA_ARCHITECTURES=native'], check=True)
subprocess.run(['cmake','--build',BUILD,'-j','2','--target','nsos_ext'], check=True)
so = glob.glob(f'{BUILD}/**/nsos_ext*.so', recursive=True)
assert so, 'nsos_ext nao compilou'
sys.path.insert(0, str(pathlib.Path(so[0]).parent))
import nsos_ext as nsos
print('nsos_ext OK:', so[0])

## 3 — Build do OxtaMem nativo (Rust + maturin) e SelectiveWriter

In [ ]:
import shutil
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
if shutil.which('cargo') is None:
    print('instalando rust toolchain...')
    subprocess.run('curl --proto =https --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y -q', shell=True, check=True)
    os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
subprocess.run([sys.executable,'-m','pip','install','-q','maturin'], check=True)
subprocess.run(['maturin','build','--release','--interpreter',sys.executable],
               cwd=f'{REPO}/modules/oxtamem', check=True)
whl = glob.glob(f'{REPO}/modules/oxtamem/oxta_engine/target/wheels/oxta_mem-*.whl')
assert whl, 'wheel do oxta_mem nao gerado'
subprocess.run([sys.executable,'-m','pip','install','-q','--force-reinstall','--no-deps',whl[0]], check=True)
from oxta_mem.native import PyGeodesicEngine
from oxta_mem.selective import SelectiveWriter
print('OxtaMem nativo OK:', whl[0])

## 4 — Tarefa de memoria (MQAR) + treino do modelo

O modelo aprende *lookup associativo em contexto*: dado `k1 v1 k2 v2 ... [Q] kq`,
prever `vq`. E a capacidade que a memoria vai estender alem da janela.

In [ ]:
import numpy as np, random, math
GPU = nsos.Device.GPU
os.environ['NSOS_ATTN_LAST_GUARD'] = '0'  # hibrido [M,A,M,A] (atencao ajuda recall)

# vocab: 0=PAD, 1=QUERY; values em [V0,V1) (byte-safe <256); keys em [K0,K1) (muitas, p/ N grande)
QUERY = 1; V0, V1 = 2, 66; K0, K1 = 66, 1266; VOCAB = 1266
D_MODEL, D_STATE = 128, 64; N_PAIRS = 8; BATCH = 32; LR = 2e-3; STEPS = 2500; SEED = 0

def _set(c,k,v):
    try: setattr(c,k,v)
    except Exception: pass
def build():
    c = nsos.ModelConfig(); c.num_layers=4; c.d_model=D_MODEL; c.vocab_size=VOCAB
    c.n_heads=4; c.n_kv_heads=2; c.max_context_tokens=1024
    c.use_moe=False; c.use_kan=False; c.use_ttt=False; c.dropout=0.0
    _set(c,'mamba2_faithful',True); _set(c,'mamba_expand',2); _set(c,'mamba_head_dim',64)
    _set(c,'mamba_state_expansion',D_STATE); _set(c,'mamba_n_groups',1)
    c.attention_period=2; c.attention_slot=1   # hibrido
    m = nsos.JambaModel(c, GPU); m.to(GPU); return m

def mqar_seq(rng, n_pairs):
    keys = rng.sample(range(K0,K1), n_pairs)
    vals = [rng.randrange(V0,V1) for _ in range(n_pairs)]
    seq = []
    for k,v in zip(keys,vals): seq += [k,v]
    qi = rng.randrange(n_pairs)
    seq += [QUERY, keys[qi]]
    return seq, vals[qi]

def batch(rng, n_pairs=N_PAIRS):
    ps, an = [], []
    for _ in range(BATCH):
        s, v = mqar_seq(rng, n_pairs); ps.append(s); an.append([v])
    return ps, an

nsos.set_seed(SEED); model = build()
tr = nsos.Trainer(model, LR); tr.warmup_steps=100; tr.total_training_steps=STEPS
tr.phase_scheduler.progressive_qat_enabled=False
rng = random.Random(7); model.set_training_mode(True)
t0=__import__('time').time(); run=0.0
for s in range(STEPS):
    npair = rng.randint(1, N_PAIRS)  # comprimento variavel: robusto a contexto curto (1 fato injetado) E cheio
    ps,an = batch(rng, npair); run += tr.train_supervised_batch(ps,an)
    if (s+1)%500==0:
        print(f'  step {s+1}/{STEPS} loss~{run/500:.3f}'); run=0.0
model.set_training_mode(False)
print(f'treino ok em {__import__("time").time()-t0:.0f}s')

## 5 — Sanity: o modelo aprendeu o lookup em contexto?

In [ ]:
def argmax_last(ids):
    lg = np.asarray(model.forward_ids(ids).cpu().numpy()).reshape(-1, VOCAB)
    return int(lg[-1].argmax())
vr = random.Random(999); ok=0; T=300
for _ in range(T):
    s,v = mqar_seq(vr, N_PAIRS)
    ok += (argmax_last(s)==v)
print(f'MQAR in-context acc (n_pairs={N_PAIRS}) = {ok/T:.3f}')

## 6 — Prova end-to-end: modelo-so vs modelo+OxtaMem (fatos >> contexto)

N fatos distintos, com C = quantos cabem na janela. **Modelo-so** so ve os ultimos
C fatos. **Modelo+OxtaMem** guarda todos os N (embedding=`forward_trunk` da chave),
recupera o fato certo e injeta 1 fato no contexto. Metrica: acc de resposta.

In [ ]:
def embed_key(k):
    tr_ = np.asarray(model.forward_trunk([QUERY, k]).cpu().numpy()).reshape(-1, D_MODEL)
    return tr_[-1].astype(np.float32).tolist()

def answer(seq):
    return argmax_last(seq)

def run_eval(N, C, n_query=200, seed=0):
    rng = random.Random(seed)
    keys = rng.sample(range(K0, K1), min(N, K1-K0))
    N = len(keys); vals = {k: rng.randrange(V0,V1) for k in keys}
    # store TODOS os fatos no OxtaMem
    db='/content/qa.db'
    for f in (db, db.replace('.db','.meta')):
        if os.path.exists(f): os.remove(f)
    eng = PyGeodesicEngine(db, 128); eng.set_sync_on_write(False)
    for k in keys:
        eng.write_with_vector(f'k{k}', bytes([vals[k]]), embed_key(k))
    eng.flush()
    # ordem de chegada (recencia): os ultimos C na 'janela' do modelo-so
    order = keys[:]  # keys[i] = i-esimo fato inserido
    qrng = random.Random(seed*7+1)
    qs = [qrng.randrange(N) for _ in range(n_query)]
    alone_ok = 0; oxta_ok = 0; retr_ok = 0
    for qi in qs:
        qk = order[qi]; true_v = vals[qk]
        # modelo-so: contexto = ultimos C fatos + query
        window = order[max(0, N-C):]
        seq = []
        for k in window: seq += [k, vals[k]]
        seq += [QUERY, qk]
        alone_ok += (answer(seq) == true_v)
        # modelo+oxta: recupera o fato certo e injeta 1 fato
        got = eng.search_similar(embed_key(qk), 1)
        rv = got[0][0] if got else -1  # value byte recuperado
        retr_ok += (rv == true_v)
        inj = [qk, rv, QUERY, qk]
        oxta_ok += (answer(inj) == true_v)
    return dict(N=N, C=C, alone=alone_ok/len(qs), oxta=oxta_ok/len(qs), retr=retr_ok/len(qs))

print(f'{"N":>6} {"C(janela)":>10} {"modelo-so":>10} {"modelo+OxtaMem":>15} {"(retr@1)":>10}')
for N in (50, 200, 1000):
    r = run_eval(N, C=8)  # janela do modelo-so = 8 fatos (= tamanho treinado, in-distribution)
    print(f'{r["N"]:>6} {r["C"]:>10} {r["alone"]:>10.3f} {r["oxta"]:>15.3f} {r["retr"]:>10.3f}')
print('\nmodelo+OxtaMem deve manter alto conforme N cresce; modelo-so cai ~C/N')

## 7 — Leitura

- **modelo-so** so acerta quando o fato consultado caiu na janela dos ultimos C ->
  acuracia ~ C/N, despenca conforme N cresce.
- **modelo+OxtaMem** recupera o fato certo da memoria O(N) e o injeta -> acuracia
  ~constante (limitada pela qualidade do retrieval, esperado ~1.0 p/ chave exata).
- Se `modelo+OxtaMem >> modelo-so` em N grande: **o OxtaMem faz o modelo responder
  melhor** — prova de qualidade end-to-end, nao so de capacidade.